In [41]:
!pip install open_clip_torch huggingface_hub semopy pykrige -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.7 MB/s eta 0:00:00 0:00:01


In [ ]:
EMBED_PATH = INPUT / "datasets/edwardsx/geovision-sit3-embeddings"
if not EMBED_PATH.exists():
    EMBED_PATH = INPUT / "geovision-sit3-embeddings"

if EMBED_PATH.exists():
    print("Cargando embeddings desde dataset...")
    data = torch.load(EMBED_PATH / "embeddings_sit3.pt", map_location="cpu", weights_only=True)
    emb = data["embeddings"]
    meta = data["meta"]
    tile_fechas = data["tile_fechas"]
    CACHE_EXISTS = True
else:
    print("Dataset no encontrado. Se generaran embeddings desde cero.")
    CACHE_EXISTS = False

In [12]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [13]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

INPUT = Path("/kaggle/input")
PANEL_PATH = INPUT / "datasets/juanjoseorozcolopez/geovision-fuentes"
if not PANEL_PATH.exists():
    PANEL_PATH = INPUT / "geovision-fuentes"

TILES_PATH = INPUT / "datasets/edwardsx/geovision-tiles-sit2"
if not TILES_PATH.exists():
    TILES_PATH = INPUT / "geovision-tiles-sit2"

MODEL_PATH = INPUT / "datasets/edwardsx/geovision-clip-modelo-v2"
if not MODEL_PATH.exists():
    MODEL_PATH = INPUT / "geovision-clip-modelo-v2"

OUTPUT = Path("/kaggle/working")
print(f"Panel: {PANEL_PATH}")
print(f"Tiles: {TILES_PATH}")
print(f"Modelo: {MODEL_PATH}")

CONTAMINANTES = ["NO2", "SO2", "O3"]

# Umbral de cobertura minima para incluir estacion en LOO-CV
COBERTURA_MINIMA = 0.20

# ConvLSTM params
CONV_HIDDEN = 128
CONV_KERNEL = 3
CONV_LAYERS = 2
CONV_LR = 1e-4
CONV_EPOCHS = 30
CONV_BATCH = 16

# Kriging params
KRIGING_VARIOGRAM = "exponential"

Device: cuda
Panel: /kaggle/input/datasets/juanjoseorozcolopez/geovision-fuentes
Tiles: /kaggle/input/datasets/edwardsx/geovision-tiles-sit2
Modelo: /kaggle/input/datasets/edwardsx/geovision-clip-modelo-v2


## Cargar DAGMA

In [14]:
pq = pd.read_parquet(PANEL_PATH / "dagma" / "dagma_cvc_horario_raw.parquet")
estaciones = pd.read_csv(PANEL_PATH / "dagma" / "estaciones_metadata.csv")

print(f"Mediciones: {len(pq)}")
print(f"Rango temporal: {pq['med_fecha_inicio'].min()} a {pq['med_fecha_inicio'].max()}")
print(f"Estaciones: {len(estaciones)}")
print()

print("Estaciones y contaminantes:")
tabla = pq.groupby(["nombre_est", "msfl_code"]).size().unstack(fill_value=0)
print(tabla.to_string())


Mediciones: 107291
Rango temporal: 2020-01-01 00:00:00 a 2024-12-31 23:00:00
Estaciones: 10

Estaciones y contaminantes:
msfl_code               NO2     O3    SO2
nombre_est                               
BASE AÉREA                0   7310   7770
CAÑAVERALEJO              0      0   5053
COMPARTIR                 0   7507      0
ERA OBRERO                0   6939      0
ESTACIÓN YUMBO         6246  16091  12035
LA ERMITA                 0      0   8627
LA FLORA                  0   5902   4165
PANCE                     0   5861      0
TRANSITORIA-NAVARRO       0   4138   3297
UNIVERSIDAD DEL VALLE     0   6350      0


In [15]:
pq_util = pq[pq["med_fecha_inicio"].copy() >= "2021-01-01"]
print(f"Mediciones en periodo util (2021-2024): {len(pq_util)}")
print()
print("Cobertura por estacion en periodo util (2021-2024, 48 meses):")
pq_util["ano_mes"] = pq_util["med_fecha_inicio"].dt.to_period("M")
for est in pq_util["nombre_est"].unique():
    mask = pq_util["nombre_est"] == est
    meses = pq_util.loc[mask, "ano_mes"].nunique()
    print(f"  {est:30s} {meses:2d}/48 ({meses/48*100:5.1f}%)")

Mediciones en periodo util (2021-2024): 55099

Cobertura por estacion en periodo util (2021-2024, 48 meses):
  BASE AÉREA                     15/48 ( 31.2%)
  CAÑAVERALEJO                    5/48 ( 10.4%)
  COMPARTIR                      13/48 ( 27.1%)
  ERA OBRERO                      8/48 ( 16.7%)
  ESTACIÓN YUMBO                 40/48 ( 83.3%)
  LA ERMITA                      10/48 ( 20.8%)
  LA FLORA                        6/48 ( 12.5%)
  PANCE                          10/48 ( 20.8%)
  UNIVERSIDAD DEL VALLE          17/48 ( 35.4%)


/tmp/ipykernel_57/41322899.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pq_util["ano_mes"] = pq_util["med_fecha_inicio"].dt.to_period("M")


## Generar secuencias ConvLSTM

In [16]:
meta = pd.read_parquet(TILES_PATH / "tiles_meta.parquet")
tile_fechas = sorted(set(pd.to_datetime([f[:8] for f in meta["time_s2"].unique()]).date))
print(f"Tiles fechas disponibles: {len(tile_fechas)}")
print(f"Rango: {tile_fechas[0]} a {tile_fechas[-1]}")


Tiles fechas disponibles: 77
Rango: 2021-01-16 a 2025-12-31


In [17]:
print("Secuencias disponibles por estacion y contaminante:")
total_secuencias = 0
for cont in CONTAMINANTES:
    datos = pq_util[pq_util["msfl_code"] == cont]
    for est in datos["nombre_est"].unique():
        est_datos = datos[datos["nombre_est"] == est].copy()
        est_datos["fecha"] = est_datos["med_fecha_inicio"].dt.date
        fechas_est = sorted(est_datos["fecha"].unique())
        seq = 0
        for f in fechas_est:
            prev = [t for t in tile_fechas if t < f]
            if len(prev) >= 8:
                seq += 1
        if seq > 0:
            print(f"  {est:30s} {cont:3s} {seq:3d} secuencias")
            total_secuencias += seq
print(f"\nTotal secuencias: {total_secuencias}")


Secuencias disponibles por estacion y contaminante:
  ESTACIÓN YUMBO                 NO2 108 secuencias
  BASE AÉREA                     SO2 166 secuencias
  CAÑAVERALEJO                   SO2  90 secuencias
  ESTACIÓN YUMBO                 SO2 629 secuencias
  LA ERMITA                      SO2 148 secuencias
  BASE AÉREA                     O3  250 secuencias
  COMPARTIR                      O3  194 secuencias
  ERA OBRERO                     O3  110 secuencias
  ESTACIÓN YUMBO                 O3  675 secuencias
  LA FLORA                       O3   93 secuencias
  PANCE                          O3  135 secuencias
  UNIVERSIDAD DEL VALLE          O3  241 secuencias

Total secuencias: 2839


## Cargar modelo CLIP + generar embeddings

In [18]:
import open_clip
from huggingface_hub import hf_hub_download

# Clases LoRA (misma estructura que el entrenamiento)
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=16):
        super().__init__()
        self.linear = linear
        d, k = linear.weight.shape
        self.A = nn.Parameter(torch.randn(d, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, k))
    @property
    def weight(self): return self.linear.weight
    @property
    def bias(self): return self.linear.bias
    def forward(self, x):
        return self.linear(x) + F.linear(x, self.A @ self.B)

def aplicar_lora(module, rank=16):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear) and name in {"out_proj", "c_fc", "c_proj"}:
            setattr(module, name, LoRALinear(child, rank))
        else:
            aplicar_lora(child, rank)

In [19]:
ckpt = torch.load(MODEL_PATH / "clip_finetuned_best.pt", map_location="cpu", weights_only=True)

clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained=None)

# Adaptar conv1 3ch -> 12ch
orig = clip_model.visual.conv1
new_conv = nn.Conv2d(12, orig.out_channels, orig.kernel_size, stride=orig.stride, bias=False)
with torch.no_grad():
    w = new_conv.weight.data
    w[:, 0] = orig.weight[:, 1]  # B4 -> R
    w[:, 1] = orig.weight[:, 2]  # B3 -> G
    w[:, 2] = orig.weight[:, 0]  # B2 -> B
    for b in range(3, 12):
        w[:, b] = orig.weight.mean(dim=1) * (3.0 / 12)
clip_model.visual.conv1 = new_conv

# Aplicar LoRA
aplicar_lora(clip_model.visual.transformer.resblocks[6:])
aplicar_lora(clip_model.transformer.resblocks[6:])

# Cargar pesos
clip_model.load_state_dict(ckpt["clip"], strict=False)
clip_model = clip_model.to(DEVICE).eval()

# Fusion
class VisualProj(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(512, 512)
    def forward(self, x): return self.proj(x)
fusion = VisualProj()
fusion.load_state_dict(ckpt["fusion"])
fusion = fusion.to(DEVICE).eval()

print("Modelo + Fusion cargados")

Modelo + Fusion cargados


In [20]:
IDX_OPTICAS = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
TILE_PX_CLIP = 224
BATCH = 64

# Normalizacion bandas
tiles_arr = np.load(TILES_PATH / "tiles_train.npz", allow_pickle=False)["data"]
flat = tiles_arr[:, IDX_OPTICAS].reshape(len(tiles_arr), 12, -1)
BAND_MEAN = flat.mean(axis=(0, 2)).astype(np.float32)
BAND_STD = flat.std(axis=(0, 2)).astype(np.float32) + 1e-6

@torch.no_grad()
def generar_embeddings():
    embs = []
    for i in tqdm(range(0, len(tiles_arr), BATCH), desc="Embeddings"):
        x = tiles_arr[i:i+BATCH, IDX_OPTICAS].astype(np.float32)
        x = (x - BAND_MEAN[None, :, None, None]) / BAND_STD[None, :, None, None]
        x = np.clip(x, -3.0, 3.0)
        x = torch.from_numpy(x).to(DEVICE)
        x = F.interpolate(x, size=TILE_PX_CLIP, mode="bilinear", align_corners=False)
        fused = F.normalize(fusion(clip_model.encode_image(x)), dim=-1)
        embs.append(fused.cpu())
    return torch.cat(embs)

emb = generar_embeddings()
print(f"Embeddings: {emb.shape}")


Embeddings:   0%|          | 0/79 [00:00<?, ?it/s]

Embeddings: torch.Size([5000, 512])


## Construir secuencias ConvLSTM

In [21]:
meta["fecha_dt"] = pd.to_datetime(meta["time_s2"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
tile_fechas = sorted(set(meta["fecha_dt"].dt.date))
idx_por_fecha = {f: meta[meta["fecha_dt"].dt.date == f].index.values for f in tile_fechas}
print(f"Fechas tile: {len(tile_fechas)}, Embeddings: {emb.shape}")

Fechas tile: 77, Embeddings: torch.Size([5000, 512])


In [22]:
from collections import defaultdict

secuencias = defaultdict(list)
contador = 0

for cont in CONTAMINANTES:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    for est in datos_cont["nombre_est"].unique():
        est_datos = datos_cont[datos_cont["nombre_est"] == est].copy()
        est_datos["fecha"] = est_datos["med_fecha_inicio"].dt.date
        for fecha, grupo in est_datos.groupby("fecha"):
            prev = [t for t in tile_fechas if t < fecha]
            if len(prev) >= 8:
                ultimas_8 = prev[-8:]
                # Verificar que haya tiles para esas 8 fechas
                hay_tiles = all(len(idx_por_fecha.get(f, [])) > 0 for f in ultimas_8)
                if hay_tiles:
                    contador += 1
                    if contador <= 5:
                        print(f"  {est:30s} {cont:3s} fecha={fecha} -> 8 tiles prev: {ultimas_8[0]} a {ultimas_8[-1]}")

print(f"\nTotal secuencias validas: {contador}")

  ESTACIÓN YUMBO                 NO2 fecha=2021-08-04 -> 8 tiles prev: 2021-01-16 a 2021-07-20
  ESTACIÓN YUMBO                 NO2 fecha=2021-08-05 -> 8 tiles prev: 2021-01-16 a 2021-07-20
  ESTACIÓN YUMBO                 NO2 fecha=2021-08-06 -> 8 tiles prev: 2021-01-16 a 2021-07-20
  ESTACIÓN YUMBO                 NO2 fecha=2021-08-07 -> 8 tiles prev: 2021-01-16 a 2021-07-20
  ESTACIÓN YUMBO                 NO2 fecha=2021-08-08 -> 8 tiles prev: 2021-01-16 a 2021-07-20

Total secuencias validas: 2839


In [23]:
torch.save({
    "embeddings": emb.cpu(),
    "meta": meta,
    "tile_fechas": tile_fechas,
}, OUTPUT / "embeddings_sit3.pt")
print(f"Embeddings guardados: {OUTPUT / 'embeddings_sit3.pt'}")
print(f"  embeddings: {emb.shape}")
print(f"  meta rows: {len(meta)}")

Embeddings guardados: /kaggle/working/embeddings_sit3.pt
  embeddings: torch.Size([5000, 512])
  meta rows: 5000


In [28]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i); f = torch.sigmoid(f)
        o = torch.sigmoid(o); g = torch.tanh(g)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new

In [29]:
class ConvLSTM(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, kernel_size=3, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            ConvLSTMCell(input_dim if i == 0 else hidden_dim, hidden_dim, kernel_size)
            for i in range(num_layers)
        ])
        self.output_conv = nn.Conv2d(hidden_dim, 3, kernel_size=1)  # 3 contaminantes

    def forward(self, x):
        b, t, c, h, w = x.shape
        for layer_idx, cell in enumerate(self.layers):
            h_t = torch.zeros(b, cell.hidden_dim, h, w, device=x.device)
            c_t = torch.zeros(b, cell.hidden_dim, h, w, device=x.device)
            outputs = []
            for t_idx in range(t):
                h_t, c_t = cell(x[:, t_idx], h_t, c_t)
                outputs.append(h_t)
            x = torch.stack(outputs, dim=1)
        return self.output_conv(x[:, -1])  # solo ultimo paso -> (b, 3, h, w)

In [30]:
model = ConvLSTM(input_dim=512, hidden_dim=128, kernel_size=3, num_layers=2).to(DEVICE)
print(f"ConvLSTM: {sum(p.numel() for p in model.parameters())/1024:.0f}K params")

ConvLSTM: 4033K params


In [31]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

batch_size = 16
seq_len = 8
input_dim = 512

# Datos sinteticos para prueba arquitectura
x_dummy = torch.randn(batch_size, seq_len, input_dim, 1, 1).to(DEVICE)
y_dummy = torch.randn(batch_size, 3, 1, 1).to(DEVICE)

for epoch in range(5):
    pred = model(x_dummy)
    loss = loss_fn(pred, y_dummy)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}: loss={loss.item():.6f}")

print("\nArquitectura ConvLSTM OK. Listo para datos reales.")

Epoch 1: loss=0.956139
Epoch 2: loss=0.954367
Epoch 3: loss=0.952587
Epoch 4: loss=0.950776
Epoch 5: loss=0.948912

Arquitectura ConvLSTM OK. Listo para datos reales.


In [35]:
meta["fecha_dt"] = pd.to_datetime(meta["time_s2"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
tile_fechas = sorted(set(meta["fecha_dt"].dt.date))
idx_por_fecha = {f: meta[meta["fecha_dt"].dt.date == f].index.values for f in tile_fechas}

X_seqs, y_vals, cont_ids = [], [], []
for cont in CONTAMINANTES:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    for est in datos_cont["nombre_est"].unique():
        est_datos = datos_cont[datos_cont["nombre_est"] == est].copy()
        est_datos["fecha"] = est_datos["med_fecha_inicio"].dt.date
        for fecha, grupo in est_datos.groupby("fecha"):
            prev = [t for t in tile_fechas if t < fecha]
            if len(prev) >= 8:
                ultimas_8 = prev[-8:]
                idxs = [idx_por_fecha[f] for f in ultimas_8]
                idxs = [i[0] for i in idxs if len(i) > 0]
                if len(idxs) == 8:
                    seq = emb[idxs].numpy()
                    val = grupo["med_concentracion_estandar"].mean()
                    X_seqs.append(seq)
                    y_vals.append(val)
                    cont_ids.append(cont)

X_seqs = np.array(X_seqs)
y_vals = np.array(y_vals, dtype=np.float32)
cont_ids = np.array(cont_ids)
# Normalizar y por contaminante
y_mean = {c: y_vals[cont_ids == c].mean() for c in CONTAMINANTES}
y_std = {c: y_vals[cont_ids == c].std() + 1e-6 for c in CONTAMINANTES}
y_norm = np.array([(y_vals[i] - y_mean[cont_ids[i]]) / y_std[cont_ids[i]] for i in range(len(y_vals))])
print(f"Dataset: X={X_seqs.shape}, y_norm std={y_norm.std():.3f}")

Dataset: X=(2839, 8, 512), y_norm std=1.000


In [37]:
cont_map = {c: j for j, c in enumerate(CONTAMINANTES)}
cont_idx = np.array([cont_map[c] for c in cont_ids])

X_t = torch.FloatTensor(X_seqs).unsqueeze(-1).unsqueeze(-1)
y_t = torch.FloatTensor(y_norm).unsqueeze(-1).unsqueeze(-1)

dataset = torch.utils.data.TensorDataset(X_t, y_t, torch.LongTensor(cont_idx))
loader = DataLoader(dataset, batch_size=16, shuffle=True)

model.train()
optim = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(10):
    total = 0
    for xb, yb, cid in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)  # (B, 3, 1, 1)
        pred_sel = pred[range(len(cid)), cid]
        loss = F.mse_loss(pred_sel.squeeze(), yb[:, 0, 0])
        optim.zero_grad(); loss.backward(); optim.step()
        total += loss.item()
    print(f"Epoch {epoch+1:2d}: loss={total/len(loader):.4f}")


Epoch  1: loss=0.9105
Epoch  2: loss=0.7171
Epoch  3: loss=0.6636
Epoch  4: loss=0.6418
Epoch  5: loss=0.6170
Epoch  6: loss=0.6112
Epoch  7: loss=0.6015
Epoch  8: loss=0.5963
Epoch  9: loss=0.6013
Epoch 10: loss=0.5927


In [38]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

meta["fecha_dt"] = pd.to_datetime(meta["time_s2"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
tile_fechas = sorted(set(meta["fecha_dt"].dt.date))
idx_por_fecha = {f: meta[meta["fecha_dt"].dt.date == f].index.values for f in tile_fechas}

resultados = []
for cont in CONTAMINANTES:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    estaciones_activas = datos_cont["nombre_est"].unique()
    cont_map = {c: j for j, c in enumerate(CONTAMINANTES)}

    for est_test in estaciones_activas:
        train_data = datos_cont[datos_cont["nombre_est"] != est_test]
        test_data = datos_cont[datos_cont["nombre_est"] == est_test]

        X_tr, y_tr = [], []
        for _, grupo in train_data.groupby(train_data["med_fecha_inicio"].dt.date):
            fecha = grupo["med_fecha_inicio"].iloc[0].date()
            prev = [t for t in tile_fechas if t < fecha]
            if len(prev) >= 8:
                idxs = [idx_por_fecha[f][0] for f in prev[-8:] if len(idx_por_fecha.get(f, [])) > 0]
                if len(idxs) == 8:
                    X_tr.append(emb[idxs].numpy())
                    y_tr.append(grupo["med_concentracion_estandar"].mean())

        if len(X_tr) < 10:
            continue

        X_tr = torch.FloatTensor(np.array(X_tr)).unsqueeze(-1).unsqueeze(-1)
        y_tr = torch.FloatTensor(y_tr).to(DEVICE)

        m = ConvLSTM().to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
        for ep in range(15):
            idxs = torch.randperm(len(X_tr))[:16]
            pred = m(X_tr[idxs].to(DEVICE))[:, cont_map[cont], 0, 0]
            loss = F.mse_loss(pred, y_tr[idxs])
            opt.zero_grad(); loss.backward(); opt.step()

        # Test
        X_te, y_te = [], []
        for _, grupo in test_data.groupby(test_data["med_fecha_inicio"].dt.date):
            fecha = grupo["med_fecha_inicio"].iloc[0].date()
            prev = [t for t in tile_fechas if t < fecha]
            if len(prev) >= 8:
                idxs = [idx_por_fecha[f][0] for f in prev[-8:] if len(idx_por_fecha.get(f, [])) > 0]
                if len(idxs) == 8:
                    X_te.append(emb[idxs].numpy())
                    y_te.append(grupo["med_concentracion_estandar"].mean())

        if len(X_te) > 0:
            X_te = torch.FloatTensor(np.array(X_te)).unsqueeze(-1).unsqueeze(-1).to(DEVICE)
            y_te = np.array(y_te)
            y_pred = m(X_te)[:, cont_map[cont], 0, 0].detach().cpu().numpy()
            rmse = np.sqrt(mean_squared_error(y_te, y_pred))
            mae = mean_absolute_error(y_te, y_pred)
            r2 = r2_score(y_te, y_pred)
            resultados.append({"estacion": est_test, "cont": cont, "n_train": len(X_tr), "n_test": len(X_te), "RMSE": rmse, "MAE": mae, "R2": r2})
            print(f"{est_test:30s} {cont:3s} | train={len(X_tr):3d} test={len(X_te):2d} | RMSE={rmse:.2f} MAE={mae:.2f} R2={r2:.2f}")

print("\nLOO-CV completado")

BASE AÉREA                     SO2 | train=784 test=166 | RMSE=8.60 MAE=7.52 R2=-3.24
CAÑAVERALEJO                   SO2 | train=858 test=90 | RMSE=2.53 MAE=2.38 R2=-8.04
ESTACIÓN YUMBO                 SO2 | train=299 test=629 | RMSE=16.49 MAE=12.50 R2=-1.35
LA ERMITA                      SO2 | train=799 test=148 | RMSE=2.59 MAE=2.40 R2=-5.93
BASE AÉREA                     O3  | train=919 test=250 | RMSE=10.02 MAE=9.24 R2=-5.64
COMPARTIR                      O3  | train=917 test=194 | RMSE=14.95 MAE=13.90 R2=-6.36
ERA OBRERO                     O3  | train=904 test=110 | RMSE=15.40 MAE=13.95 R2=-4.56
ESTACIÓN YUMBO                 O3  | train=435 test=675 | RMSE=24.36 MAE=19.48 R2=-1.77
LA FLORA                       O3  | train=920 test=93 | RMSE=18.16 MAE=16.74 R2=-5.64
PANCE                          O3  | train=917 test=135 | RMSE=28.23 MAE=25.98 R2=-5.54
UNIVERSIDAD DEL VALLE          O3  | train=878 test=241 | RMSE=14.14 MAE=11.84 R2=-2.34

LOO-CV completado


In [39]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings; warnings.filterwarnings("ignore")

meta["fecha_dt"] = pd.to_datetime(meta["time_s2"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
tile_fechas = sorted(set(meta["fecha_dt"].dt.date))

era5_cols = ["era5_T2m", "era5_Td2m", "era5_u10", "era5_v10", "era5_BLH", "era5_RH850", "era5_psurf", "era5_precip"]

resultados = []
for cont in CONTAMINANTES:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    estaciones_activas = datos_cont["nombre_est"].unique()

    for est_test in estaciones_activas:
        test_data = datos_cont[datos_cont["nombre_est"] == est_test]
        train_data = datos_cont[datos_cont["nombre_est"] != est_test]

        def build_dataset(df):
            X, y = [], []
            for _, grupo in df.groupby(df["med_fecha_inicio"].dt.date):
                fecha = grupo["med_fecha_inicio"].iloc[0].date()
                prev = [t for t in tile_fechas if t < fecha]
                if len(prev) >= 1:
                    ult = meta[meta["fecha_dt"].dt.date == prev[-1]]
                    if len(ult) > 0:
                        i = ult.index[0]
                        feat = np.concatenate([emb[i].numpy(), meta.loc[i, era5_cols].values.astype(float)])
                        X.append(feat)
                        y.append(grupo["med_concentracion_estandar"].mean())
            return np.array(X), np.array(y)

        X_tr, y_tr = build_dataset(train_data)
        X_te, y_te = build_dataset(test_data)

        if len(X_tr) < 10 or len(X_te) < 3:
            continue

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)

        model = Ridge(alpha=1.0)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        mae = mean_absolute_error(y_te, y_pred)
        r2 = r2_score(y_te, y_pred)
        resultados.append({"estacion": est_test, "cont": cont, "n_train": len(X_tr), "n_test": len(X_te), "RMSE": rmse, "MAE": mae, "R2": r2})
        print(f"{est_test:30s} {cont:3s} | train={len(X_tr):3d} test={len(X_te):3d} | RMSE={rmse:.2f} MAE={mae:.2f} R2={r2:.2f}")

print(f"\nR2 medio={np.mean([r["R2"] for r in resultados]):.3f}")

BASE AÉREA                     SO2 | train=784 test=166 | RMSE=5.01 MAE=3.45 R2=-0.44
CAÑAVERALEJO                   SO2 | train=858 test= 90 | RMSE=1.47 MAE=1.43 R2=-2.06
ESTACIÓN YUMBO                 SO2 | train=299 test=629 | RMSE=12.78 MAE=8.56 R2=-0.41
LA ERMITA                      SO2 | train=799 test=148 | RMSE=3.25 MAE=2.47 R2=-9.93
BASE AÉREA                     O3  | train=919 test=252 | RMSE=7.23 MAE=6.25 R2=-2.35
COMPARTIR                      O3  | train=919 test=194 | RMSE=7.12 MAE=5.55 R2=-0.67
ERA OBRERO                     O3  | train=906 test=110 | RMSE=7.12 MAE=5.63 R2=-0.19
ESTACIÓN YUMBO                 O3  | train=437 test=675 | RMSE=14.73 MAE=10.58 R2=-0.01
LA FLORA                       O3  | train=922 test= 93 | RMSE=7.62 MAE=5.08 R2=-0.17
PANCE                          O3  | train=919 test=135 | RMSE=18.76 MAE=15.79 R2=-1.89
UNIVERSIDAD DEL VALLE          O3  | train=880 test=241 | RMSE=9.51 MAE=7.18 R2=-0.51

R2 medio=-1.693


In [ ]:
print("\n=== Kriging Ordinario LOO-CV ===\n")

for cont in ["SO2", "O3"]:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    estaciones = datos_cont["nombre_est"].unique()
    lats, lons, vals, ests = [], [], [], []

    for est in estaciones:
        sub = datos_cont[datos_cont["nombre_est"] == est]
        if len(sub) > 10:
            lats.append(sub["latitud"].mean())
            lons.append(sub["longitud"].mean())
            vals.append(sub["med_concentracion_estandar"].mean())
            ests.append(est)

    lats = np.array(lats); lons = np.array(lons); vals = np.array(vals)
    print(f"\n{cont}: {len(lats)} estaciones")

    errores = []
    for i in range(len(lats)):
        mascara = np.ones(len(lats), dtype=bool)
        mascara[i] = False
        lat_test, lon_test = lats[i], lons[i]
        lat_train, lon_train = lats[mascara], lons[mascara]
        val_real = vals[i]
        val_train = vals[mascara]

        if len(lat_train) < 3:
            continue

        try:
            ok = OrdinaryKriging(lat_train, lon_train, val_train,
                                 variogram_model="exponential",
                                 verbose=False, enable_statistics=False)
            pred, _ = ok.execute("points", [lat_test], [lon_test])
            errores.append(abs(pred[0] - val_real))
            print(f"  {ests[i]:30s} real={val_real:6.2f} pred={pred[0]:6.2f} error={abs(pred[0]-val_real):.2f}")
        except Exception as e:
            print(f"  {ests[i]:30s} ERROR: {e}")

    if len(errores) > 0:
        print(f"  MAE Kriging: {np.mean(errores):.2f} ug/m3")
